# Create Dataset

This notebook creates a dataset containing training, testing and validation splits with options for choosing a desired amount of X % of each month of the year (provided such yearly dataset) and with X % upturning sampling of flights for better flight turning prediction.

In [ ]:
import sys
from pathlib import Path
import importlib

# Add project root to path
project_root = Path("/home/fusg/VT_2")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import from utils package
import utils
importlib.reload(utils)

from utils import (
    WindowParams,
    SplitConfig,
    SamplingConfig,
    StatsConfig,
    TurnSampling,
    build_or_load_dataset,
    collect_parquet_files,
    load_data_from_files,
)

print("Imports successful!")

Imports successful!


In [2]:
# Configuration
DATA_DIR = "/store/Projects_CRM/data_clean_modified"
DATA_FRAC = 0.20  # 20% of data from all months
TURN_UPSAMPLING = 0.15  # 15% turn upsampling

# Window parameters
wparams = WindowParams(
    input_len=60,      # 60 historical time steps (1 minute at 1Hz)
    output_horizon=60,  # 60 future time steps (1 minute at 5Hz)
    output_stride=5,    # 5 second intervals
    overlap=False       # No overlap between windows
)

# Split configuration
scfg = SplitConfig(
    train_frac=0.7,   # 70% training
    val_frac=0.15,    # 15% validation
    split_seed=42     # Reproducible splits
)

# Sampling configuration with 15% turn upsampling
samp = SamplingConfig(
    n_train=2_000_000,  # 2M training samples
    n_val=500_000,      # 500K validation samples
    n_test=400_000,     # 400K test samples
    train_turn=TurnSampling(
        min_turn_frac=TURN_UPSAMPLING,  # 15% turn upsampling
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
    val_turn=TurnSampling(
        min_turn_frac=TURN_UPSAMPLING,  # 15% turn upsampling
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
    test_turn=TurnSampling(
        min_turn_frac=0.0,  # No turn upsampling for test
        turn_thr=0.01,
        consec=3,
        consider_hist=True,
        consider_future=True
    ),
)

# Normalization statistics configuration
stats_cfg = StatsConfig(
    stats_seed=1234,
    stats_sample_size=2_000_000  # Sample size for computing normalization stats
)

print("Configuration set up!")
print(f"  Data fraction: {DATA_FRAC*100}%")
print(f"  Turn upsampling: {TURN_UPSAMPLING*100}%")

Configuration set up!
  Data fraction: 20.0%
  Turn upsampling: 15.0%


In [3]:
# Create dataset
print("Creating dataset...")
print("=" * 70)

# Step 1: Collect parquet files (20% from each month)
print("\nStep 1: Collecting parquet files...")
parquet_files = collect_parquet_files(DATA_DIR, DATA_FRAC)

# Step 2: Load and combine data
print("\nStep 2: Loading and engineering data...")
df = load_data_from_files(parquet_files)

# Step 3: Build or load dataset
print("\nStep 3: Building dataset...")
(X_train, Y_train, C_train,
 X_val, Y_val, C_val,
 X_test, Y_test, C_test,
 norm_stats, meta_train, meta_val, meta_test,
 manifest, summary) = build_or_load_dataset(
    df=df,
    wparams=wparams,
    scfg=scfg,
    samp=samp,
    stats_cfg=stats_cfg,
)

print("=" * 70)
print("Dataset creation complete!")
print(f"  Train samples: {len(X_train)}")
print(f"  Val samples: {len(X_val)}")
print(f"  Test samples: {len(X_test)}")

Creating dataset...

Step 1: Collecting parquet files...
Found 36 parquet files total
Found 12 months with parquet files
  apr: selected 1/3 files
  aug: selected 1/3 files
  dez: selected 1/3 files
  feb: selected 1/3 files
  jan: selected 1/3 files
  jul: selected 1/3 files
  jun: selected 1/3 files
  mar: selected 1/3 files
  may: selected 1/3 files
  nov: selected 1/3 files
  okt: selected 1/3 files
  sep: selected 1/3 files

Total selected: 12 files (20.0% from each month)

Step 2: Loading and engineering data...
Loading data from 12 files...
    apr02.parquet: 33,193,635 rows, 32,701 flights
    aug00.parquet: 36,006,551 rows, 35,202 flights
    dez01.parquet: 21,735,111 rows, 21,983 flights
    feb01.parquet: 20,957,805 rows, 21,575 flights
  Processed 5/12 files...
    jan02.parquet: 22,222,823 rows, 23,005 flights
    jul01.parquet: 39,938,924 rows, 38,965 flights
    jun00.parquet: 34,318,690 rows, 33,713 flights
    mar01.parquet: 23,330,743 rows, 23,568 flights
    may02.pa